# Stage 10 — Talking to the server

**Goal:** drive your model the way an application would.

`llama-server` implements the OpenAI API. That single fact is what makes the last
stage of this project short: the `openai` SDK, Continue.dev, and `curl` all work
against it with nothing but a base-URL change.

**Start the server first**, in a terminal:

```powershell
.\scripts\serve.ps1
```

Then run these cells.

In [ ]:
# --- Local bootstrap -------------------------------------------------------
# Kernel: "tinyllm (local, no torch)" -- registered by scripts\setup_local.ps1.
# There is deliberately no PyTorch in this environment.
import subprocess, sys
from pathlib import Path

REPO = Path.cwd()
while not (REPO / "pyproject.toml").exists() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO / "src"))

from tinyllm import config
from tinyllm.config import model_cfg, quant_cfg, serve_cfg, gen_cfg, hub

MODELS = REPO / "models"
VENDOR = REPO / "vendor" / "llamacpp"

def llama_tool(name):
    """Locate a llama.cpp binary; its position inside the release zip moves."""
    hits = list(VENDOR.rglob(f"{name}.exe")) or list(VENDOR.rglob(name))
    if not hits:
        raise FileNotFoundError(f"{name} not found under {VENDOR}. Run scripts\\setup_local.ps1.")
    return hits[0]

import importlib.util
assert importlib.util.find_spec("torch") is None, (
    "torch is installed in the local venv -- it should not be. See pyproject.toml."
)
print(f"repo:   {REPO}")
print(f"models: {[p.name for p in sorted(MODELS.glob('*.gguf'))] or 'none yet'}")
print("torch:  absent, by design")

In [ ]:
from openai import OpenAI

client = OpenAI(base_url=serve_cfg.base_url, api_key="none")

try:
    models = client.models.list()
    print(f"connected to {serve_cfg.base_url}")
    for m in models.data:
        print(f"  model: {m.id}")
except Exception as e:
    raise SystemExit(f"Cannot reach the server ({e}).\nStart it with: .\\scripts\\serve.ps1")

## 10.1 — A chat completion

Note what is *not* here: no prompt formatting, no special tokens, no chat
template. Messages go in as roles and content. The server applies the template it
read from the GGUF — the end of the four-hop journey that string has taken since
`config.py`.

In [ ]:
resp = client.chat.completions.create(
    model=serve_cfg.served_model_name,
    messages=[{"role": "user", "content": "Write a story about a lost puppy who finds its way home."}],
    temperature=0.8,
    max_tokens=250,
)

print(resp.choices[0].message.content)
print(f"\n---\nfinish_reason: {resp.choices[0].finish_reason}")
if resp.usage:
    print(f"prompt {resp.usage.prompt_tokens} + completion {resp.usage.completion_tokens} "
          f"= {resp.usage.total_tokens} tokens")

A `finish_reason` of `stop` means the model emitted `<|im_end|>` and the server
recognised it — the EOS decision made back in stage 7 working correctly. If you
see `length` every time, generation is running past the end of the reply until it
hits the token cap, which would mean the EOS id in the GGUF is wrong.

## 10.2 — Streaming

Tokens as they're produced, which is what makes a chat UI feel responsive rather
than frozen.

In [ ]:
import time

stream = client.chat.completions.create(
    model=serve_cfg.served_model_name,
    messages=[{"role": "user", "content": "Write a short story using the words: ball, tree, happy."}],
    temperature=0.8, max_tokens=200, stream=True,
)

t0 = time.time()
first_token_at = None
n = 0
for event in stream:
    if not event.choices:
        continue
    piece = event.choices[0].delta.content
    if piece:
        if first_token_at is None:
            first_token_at = time.time() - t0
        print(piece, end="", flush=True)
        n += 1

total = time.time() - t0
print(f"\n\n--- time to first token {first_token_at:.2f}s | "
      f"{n} tokens in {total:.2f}s ({n/total:.1f} tok/s)")

## 10.3 — Temperature, from the client side

Same request, three temperatures. This is the stage 5 experiment again, but now
through the API — the sampling happens server-side in C++ rather than in Python.

In [ ]:
prompt = "Write a story about a brave little boat."

for temp in (0.2, 0.8, 1.3):
    r = client.chat.completions.create(
        model=serve_cfg.served_model_name,
        messages=[{"role": "user", "content": prompt}],
        temperature=temp, max_tokens=140, seed=0,
    )
    print(f"=== temperature {temp}")
    print(r.choices[0].message.content[:400])
    print()

## 10.4 — Multi-turn, and the context limit

The API is stateless: the client resends the whole conversation each time. With a
**512-token context**, that budget disappears fast — which is a real constraint of
this model, and worth feeling directly.

In [ ]:
messages = [{"role": "user", "content": "Write a very short story about a cat."}]

for turn, follow_up in enumerate(["Now make it happier.", "Give the cat a name."], 1):
    r = client.chat.completions.create(
        model=serve_cfg.served_model_name, messages=messages,
        temperature=0.8, max_tokens=120,
    )
    reply = r.choices[0].message.content
    used = r.usage.total_tokens if r.usage else 0
    print(f"--- turn {turn}  (context used: {used}/{serve_cfg.ctx_size})")
    print(reply[:300])
    print()

    messages.append({"role": "assistant", "content": reply})
    messages.append({"role": "user", "content": follow_up})

print("Once the conversation exceeds 512 tokens the server truncates the front of")
print("the prompt -- the model loses the beginning of its own conversation. Real")
print("chat applications manage this with summarisation or a sliding window.")

## 10.5 — The raw HTTP

Underneath the SDK it is one POST. Useful to see once, because it's what makes
the "OpenAI-compatible" claim concrete.

In [ ]:
import requests, json

r = requests.post(
    f"{serve_cfg.base_url}/chat/completions",
    headers={"Content-Type": "application/json"},
    json={
        "model": serve_cfg.served_model_name,
        "messages": [{"role": "user", "content": "Write a story about a tiny dragon."}],
        "temperature": 0.8,
        "max_tokens": 120,
    },
    timeout=120,
)
data = r.json()
print(json.dumps({k: v for k, v in data.items() if k != "choices"}, indent=2)[:600])
print("\ncontent:\n", data["choices"][0]["message"]["content"][:400])

## 10.6 — The VS Code sidebar

Last step:

```powershell
Copy-Item continue\config.yaml $HOME\.continue\config.yaml
```

Install the **Continue** extension, reload VS Code, open the sidebar, pick
**tinyllm (local)**, and ask it for a story.

It is configured for `chat` only. A 15.7M-parameter story model cannot write
code, and giving it `autocomplete` or `edit` would insert nonsense into your
files — making a correctly-working setup feel broken. See `docs/10-vscode.md`.

---

## You're done

Trace the reply in that sidebar backwards:

```
the text in the sidebar
  ← Continue's HTTP request
  ← llama-server's sampler
  ← your Q8_0 GGUF
  ← your f16 GGUF
  ← your Hugging Face repo
  ← your SFT run
  ← your 45-minute pretraining run
  ← 328M tokens through 8 transformer blocks
  ← random initialization
```

Every one of those arrows is a stage in this repo, and you built all of them.